In [1]:
%pip install jupysql pandas matplotlib duckdb-engine
import duckdb
import pandas as pd
import build_geonames_db
import geonames_db_search
from pathlib import Path

if "conn" in globals():
    globals()["conn"].close() # Allow script rerunning

%load_ext sql
geonames_search = geonames_db_search.GeonamesSearch(topk=3, threshold=2)
conn = geonames_search.connection
%config SqlMagic.displaylimit = None
%sql conn --alias duckdb

Note: you may need to restart the kernel to use updated packages.


The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

displaylimit: Value None will be treated as 0 (no limit)

In [2]:

print(f"DB Size: {Path(build_geonames_db.DUCK_DB_PATH).stat().st_size / 1e9:.2f} GB")
description = conn.execute("""DESCRIBE""").fetchdf()
description = description[['name', 'column_names', 'column_types']]
description.set_index('name', inplace=True)
description.insert(0, 'row_count', float('nan'))


for table_name in description.index:
    if table_name not in ["fullHierarchy", "allNames"]:
        row_count = conn.execute(f'SELECT COUNT(*) FROM {table_name}').fetchone()[0]
        description.loc[table_name, 'row_count'] = row_count

display(description.style.format({'row_count': '{:_.0f}'}))

feature_codes = conn.execute(
    """
        SELECT feature_class, feature_code, COUNT(*) AS count
        FROM geonames
        GROUP BY feature_class, feature_code
        ORDER BY count DESC
""").fetchdf()
feature_codes['feature_code'] = feature_codes['feature_class'] + "." + feature_codes['feature_code']
feature_codes.drop(columns=['feature_class'], inplace=True)
feature_codes["proportion"] = feature_codes['count'] / feature_codes['count'].sum()
feature_codes = feature_codes.head(20)
try:
    build_geonames_db.download_file("https://download.geonames.org/export/dump/featureCodes_en.txt", Path("dumps/geonames/featureCodes_en.txt"))
    feature_codes = feature_codes.merge(
        pd.read_csv(
            "dumps/geonames/featureCodes_en.txt", 
            sep='\t', header=None, names=["feature_code", "designation", "description"]
    ), on='feature_code', how='left')
    feature_codes = feature_codes[['feature_code', 'designation', 'count', 'proportion']]
except Exception as e:
    print(f"Could not download feature code descriptions: {e}")
display(
    feature_codes.style
    .format({'count': '{:_.0f}', 'proportion': '{:.0%}'})
    .bar(subset=['proportion'], vmin=0, vmax=1)
)

DB Size: 1.88 GB


,row_count,column_names,column_types
name,,,
allNames,nan,['geonameId' 'isolanguage' 'alternateName' 'isPreferredName' 'isShortName' 'isColloquial' 'isHistoric' 'gndUri'],['INTEGER' 'VARCHAR' 'VARCHAR' 'BOOLEAN' 'BOOLEAN' 'BOOLEAN' 'BOOLEAN' 'VARCHAR']
alternateNames,18_769_801,['alternateNameId' 'geonameId' 'isolanguage' 'alternateName' 'isPreferredName' 'isShortName' 'isColloquial' 'isHistoric'],['INTEGER' 'INTEGER' 'VARCHAR' 'VARCHAR' 'BOOLEAN' 'BOOLEAN' 'BOOLEAN' 'BOOLEAN']
countryInfo,252,['ISO' 'ISO3' 'ISO_Numeric' 'fips' 'Country' 'Capital' 'Area' 'Population' 'Continent' 'tld' 'CurrencyCode' 'CurrencyName' 'Phone' 'Postal_Code_Format' 'Postal_Code_Regex' 'Languages' 'geonameid' 'neighbours' 'EquivalentFipsCode'],['VARCHAR' 'VARCHAR' 'INTEGER' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'FLOAT' 'INTEGER' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'INTEGER' 'VARCHAR' 'VARCHAR']
fullHierarchy,nan,['childId' 'parentId' 'level'],['INTEGER' 'INTEGER' 'VARCHAR']
geonames,13_412_879,['geonameId' 'name' 'asciiname' 'alternatenames' 'latitude' 'longitude' 'feature_class' 'feature_code' 'country_code' 'cc2' 'admin1_code' 'admin2_code' 'admin3_code' 'admin4_code' 'admin5_code' 'population' 'elevation' 'dem' 'timezone' 'modification_date'],['INTEGER' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'FLOAT' 'FLOAT' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'VARCHAR' 'BIGINT' 'INTEGER' 'INTEGER' 'VARCHAR' 'DATE']
gnd,65_617,['gndUri' 'geonameId'],['VARCHAR' 'INTEGER']
gndNames,132_370,['gndUri' 'name' 'isPreferred'],['VARCHAR' 'VARCHAR' 'BOOLEAN']
hierarchy,518_269,['parentId' 'childId' 'type'],['INTEGER' 'INTEGER' 'VARCHAR']
informalParentCity,169_815,['childId' 'parentId'],['INTEGER' 'INTEGER']


,feature_code,designation,count,proportion
0,P.PPL,populated place,4_685_832,35%
1,H.STM,stream,990_177,7%
2,T.HLL,hill,502_096,4%
3,T.MT,mountain,422_594,3%
4,S.FRM,farm,359_812,3%
5,H.LK,lake,309_547,2%
6,S.SCH,school,298_042,2%
7,H.STMI,intermittent stream,261_178,2%
8,S.CH,church,259_635,2%
9,S.HTL,hotel,241_608,2%


In [3]:
%%sql
SELECT COUNT(*) FROM geonames WHERE feature_class = 'P'

Running query in 'duckdb'

count_star()
5195947


5 195 947 populated places in geonames against [2 887 930](https://query.wikidata.org/#SELECT%20%28COUNT%28DISTINCT%20%3Fid%29%20AS%20%3Fcount%29%20WHERE%20%7B%0A%20%20%3Fid%20wdt%3AP31%20%2F%20wdt%3AP279%2a%20wd%3AQ123964505.%0A%7D) on wikidata

However there are [709 681](https://query.wikidata.org/#SELECT%20%28COUNT%28DISTINCT%20%3Fid%29%20AS%20%3Fcount%29%20WHERE%20%7B%0A%20%20%3Fid%20wdt%3AP31%20%2F%20wdt%3AP279%2a%20wd%3AQ123964505.%0A%20%20%3Fid%20wdt%3AP1566%20%3FsomeGeonamesId%0A%7D) wikidata populated places that don't link to a geoname.

Among which [13 554](https://query.wikidata.org/#SELECT%20%28COUNT%28DISTINCT%20%3Fid%29%20AS%20%3Fcount%29%20WHERE%20%7B%0A%20%20%3Fid%20wdt%3AP31%20%2F%20wdt%3AP279%2a%20wd%3AQ253019.%0A%20%20%3Fid%20wdt%3AP1566%20%3FsomeGeonamesId%0A%7D) are ["Ortsteil"](https://www.wikidata.org/wiki/Q253019) including [Sudberg](https://www.wikidata.org/wiki/Q2362997) in Wuppertal, which is present in bzk open but not [geonames](https://www.geonames.org/2805753/wuppertal.html) neither gnd.

In [4]:
%%sql
SELECT levenshtein(nfc_normalize(lower(alternateName)), nfc_normalize(lower('Sudberg'))) AS score, *
FROM allNames NATURAL LEFT JOIN simplifiedGeonames
WHERE score <= 2 AND feature_class = 'P'
ORDER BY score
LIMIT 10;

Running query in 'duckdb'

score,geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri,name,asciiname,feature_class,feature_code,country_code,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,parentCityId
1,11587121,de,Surberg,None,None,None,None,None,Surberg,Surberg,P,PPLL,CH,SG,1724,3275,None,None,None
1,2824593,None,Surberg,True,None,None,None,None,Surberg,Surberg,P,PPLA4,DE,02,091,09189,09189148,None,None
1,2942109,None,Budberg,True,None,None,None,None,Budberg,Budberg,P,PPL,DE,07,051,05170,05170032,None,None
1,5214588,None,Suedberg,True,None,None,None,None,Suedberg,Suedberg,P,PPL,US,PA,107,60464,None,None,None
1,2658465,None,Suberg,True,None,None,None,None,Suberg,Suberg,P,PPL,CH,BE,243,303,None,None,None
1,11962460,se,Sundberg,True,None,None,None,None,Salmenkallio,Salmenkallio,P,PPLX,FI,01,011,091,None,None,None
1,11587121,None,Surberg,True,None,None,None,None,Surberg,Surberg,P,PPLL,CH,SG,1724,3275,None,None,None
1,2942109,None,Budberg,None,None,None,None,None,Budberg,Budberg,P,PPL,DE,07,051,05170,05170032,None,None
1,2824593,None,Surberg,True,None,None,None,https://d-nb.info/gnd/4305569-2,Surberg,Surberg,P,PPLA4,DE,02,091,09189,09189148,None,None
1,2942108,None,Budberg,True,None,None,None,None,Budberg,Budberg,P,PPL,DE,07,059,05974,05974052,None,None


In [5]:
%%sql
SELECT levenshtein(nfc_normalize(lower(alternateName)), nfc_normalize(lower('Marienfelde'))) AS score, *
FROM allNames NATURAL LEFT JOIN simplifiedGeonames
WHERE score <= 2 AND feature_class = 'P'
ORDER BY score
LIMIT 10;


Running query in 'duckdb'

score,geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri,name,asciiname,feature_class,feature_code,country_code,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,parentCityId
0,3099326,None,Marienfelde,None,None,None,None,None,Glaznoty,Glaznoty,P,PPL,PL,85,2815,281509,None,None,None
0,2873591,None,Marienfelde,None,None,None,None,None,Marienfelde,Marienfelde,P,PPL,DE,12,00,13075,13075070,None,None
0,2873589,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPLX,DE,16,00,11000,11000000,07,2950159
0,2873589,None,Marienfelde,True,None,None,None,https://d-nb.info/gnd/5203641-8,Marienfelde,Marienfelde,P,PPLX,DE,16,00,11000,11000000,07,2950159
0,2873592,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPL,DE,12,00,13075,13075130,None,None
0,3088128,None,Marienfelde,None,None,None,None,None,Prątnik,Pratnik,P,PPL,PL,85,2802,280202,None,None,None
0,2873591,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPL,DE,12,00,13075,13075070,None,None
0,2873593,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPL,DE,12,00,13071,13071124,None,None
0,3092300,None,Marienfelde,None,None,None,None,None,Marianka,Marianka,P,PPL,PL,85,2804,280407,None,None,None
0,2873590,None,Marienfelde,True,None,None,None,None,Marienfelde,Marienfelde,P,PPL,DE,10,00,01058,01058157,None,None


In [ ]:
%%sql
WITH 
query_prep AS (
    SELECT 
        nfc_normalize('Lisboa') AS query_name,
        regexp_replace(lower(strip_accents(nfc_normalize('Lisboa'))), '[^\w\s]', '', 'g') AS clean_query_name,
        '.' IN query_name AS check_for_abbreviation,
        replace(lower(strip_accents(query_name)), '.', '\w+\s*') AS abbreviation_pattern
),
ranked_matches AS(
    SELECT 
        nfc_normalize(alternateName) AS match_name,
        nfc_normalize('Lisboa') AS query_name,
        regexp_replace(lower(strip_accents(match_name)), '[^\w\s]', '', 'g') AS clean_match_name,
        regexp_replace(lower(strip_accents(query_name)), '[^\w\s]', '', 'g') AS clean_query_name,
        levenshtein(match_name, query_name) AS raw_distance,
        levenshtein(clean_match_name, clean_query_name) AS cleaned_distance,
        CASE WHEN '.' IN query_name THEN clean_match_name SIMILAR TO replace(lower(strip_accents(query_name)), '.', '\w+\s*') ELSE FALSE END AS may_be_abbreviation,
        ROW_NUMBER() OVER (
            PARTITION BY geonameId 
            ORDER BY 
                CASE 
                    WHEN raw_distance = 0 THEN 0 
                    WHEN cleaned_distance = 0 THEN 1
                    WHEN may_be_abbreviation THEN 2 
                    ELSE 3 
                END,
                raw_distance, 
                cleaned_distance,
                CASE WHEN isPreferredName IS TRUE THEN 0 ELSE 1 END
            ASC) AS match_rank,
        allNames.*, simplifiedGeonames.*, countryInfo.Country
    FROM allNames NATURAL LEFT JOIN simplifiedGeonames LEFT JOIN countryInfo ON (country_code = ISO)
    WHERE 
        (least(raw_distance, cleaned_distance) <= 2 OR may_be_abbreviation) AND 
        feature_class = 'P' AND 
        (
            isolanguage IS NULL OR 
            isolanguage IN countryInfo.Languages OR 
            split(isolanguage, '-')[1] IN ('en', 'de')
        )
),
ranked_entities AS (
    SELECT *,
    RANK() OVER (
        ORDER BY 
            CASE 
                WHEN raw_distance = 0 THEN 0 
                WHEN cleaned_distance = 0 THEN 1
                WHEN may_be_abbreviation THEN 2 
                ELSE 3 
            END,
            raw_distance, 
            cleaned_distance,
            CASE WHEN isPreferredName IS TRUE THEN 0 ELSE 1 END
        ASC) AS entity_rank
    FROM ranked_matches
    WHERE match_rank = 1
)
SELECT * EXCLUDE (match_rank)
FROM ranked_entities
WHERE entity_rank <= 10
ORDER BY entity_rank;

Running query in 'duckdb'

match_name,query_name,clean_match_name,clean_query_name,raw_distance,cleaned_distance,may_be_abbreviation,geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri,geonameId_1,name,asciiname,feature_class,feature_code,country_code,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,parentCityId,Country,entity_rank
Lisboa,Lisboa,lisboa,lisboa,0,0,False,13293090,None,Lisboa,True,None,None,None,None,13293090,Lisboa,Lisboa,P,PPLX,MX,06,037,None,None,None,None,Mexico,1
Lisboa,Lisboa,lisboa,lisboa,0,0,False,6394277,None,Lisboa,True,None,None,None,None,6394277,Lisboa,Lisboa,P,PPLL,PE,22,2210,221005,None,None,None,Peru,1
Lisboa,Lisboa,lisboa,lisboa,0,0,False,8182114,None,Lisboa,True,None,None,None,None,8182114,Lisboa,Lisboa,P,PPLX,CO,34,11001,11,None,None,8182109,Colombia,1
Lisboa,Lisboa,lisboa,lisboa,0,0,False,2267057,pt,Lisboa,True,None,None,None,None,2267057,Lisbon,Lisbon,P,PPLC,PT,14,1106,None,None,None,None,Portugal,1
Lisboa,Lisboa,lisboa,lisboa,0,0,False,3676522,None,Lisboa,True,None,None,None,None,3676522,Lisboa,Lisboa,P,PPL,CO,28,73043,None,None,None,None,Colombia,1
Lisboa,Lisboa,lisboa,lisboa,0,0,False,1093109,None,Lisboa,True,None,None,None,None,1093109,Lisboa,Lisboa,P,PPL,MZ,09,None,None,None,None,None,Mozambique,1
Lisboa,Lisboa,lisboa,lisboa,0,0,False,1043653,None,Lisboa,True,None,None,None,None,1043653,Lisboa,Lisboa,P,PPL,MZ,10,None,None,None,None,None,Mozambique,1
Lisboa,Lisboa,lisboa,lisboa,0,0,False,3695346,None,Lisboa,True,None,None,None,None,3695346,Lisboa,Lisboa,P,PPL,PE,16,1606,160605,None,None,None,Peru,1
Lisboa,Lisboa,lisboa,lisboa,0,0,False,3732391,None,Lisboa,True,None,None,None,None,3732391,Lisboa,Lisboa,P,PPL,CO,26,68524,None,None,None,None,Colombia,1
Lisboa,Lisboa,lisboa,lisboa,0,0,False,1043652,None,Lisboa,True,None,None,None,None,1043652,Lisboa,Lisboa,P,PPL,MZ,05,None,None,None,None,None,Mozambique,1


In [15]:
import time
from utils import format_time
cities = [
    "Lisboa", "New York", "São Paulo", "München", "Berlin", "Tokyo", "London", "Paris", "Sydney",
    "Dubai", "Singapore", "Hong Kong", "Barcelona", "Amsterdam", "Rome", "Istanbul", "Moscow", "Toronto", "Chicago", "Los Angeles",
    "Madrid", "Bangkok", "Mexico City", "Seoul", "Mumbai", "Delhi", "Cairo", "Lagos", "Buenos Aires", "Santiago", "Johannesburg",
]
ESTIMATED_TOTAL_ADDRESSES = 4_228_682 # from compare.ipynb
start_time = time.monotonic()
matches = geonames_search.find_closest_matches(cities, entity_type=geonames_db_search.EntityType["City"])
end_time = time.monotonic()
elapsed = end_time - start_time
print(f"Search took {format_time(elapsed)}\nEstimate time for {ESTIMATED_TOTAL_ADDRESSES:_} addresses: {format_time((elapsed / len(cities)) * 2 * ESTIMATED_TOTAL_ADDRESSES)}")
matches

Search took 0:00:26 - Estimate time for 4_228_682 addresses: 2 months, 23 days, 14:50:11


,query_part,match_name,query_name,clean_match_name,clean_query_name,raw_distance,cleaned_distance,may_be_abbreviation,geonameId,isolanguage,...,feature_code,country_code,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,parentCityId,Country,entity_rank
0,Lisboa,Lisboa,Lisboa,lisboa,lisboa,0,0,False,3732391,NaN,...,PPL,CO,26,68524,NaN,None,None,<NA>,Colombia,1
1,Lisboa,Lisboa,Lisboa,lisboa,lisboa,0,0,False,1043654,NaN,...,PPL,MZ,09,NaN,NaN,None,None,<NA>,Mozambique,1
2,Lisboa,Lisboa,Lisboa,lisboa,lisboa,0,0,False,6394277,NaN,...,PPLL,PE,22,2210,221005,None,None,<NA>,Peru,1
3,Lisboa,Lisboa,Lisboa,lisboa,lisboa,0,0,False,3695346,NaN,...,PPL,PE,16,1606,160605,None,None,<NA>,Peru,1
4,Lisboa,Lisboa,Lisboa,lisboa,lisboa,0,0,False,13293090,NaN,...,PPLX,MX,06,037,NaN,None,None,<NA>,Mexico,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1735,Johannesburg,Johannesburg,Johannesburg,johannesburg,johannesburg,0,0,False,3383989,NaN,...,PPL,SR,11,NaN,NaN,None,None,<NA>,Suriname,1
1736,Johannesburg,Johannesburg,Johannesburg,johannesburg,johannesburg,0,0,False,4997570,NaN,...,PPL,US,MI,137,14840,None,None,<NA>,United States,1
1737,Johannesburg,Johannesburg,Johannesburg,johannesburg,johannesburg,0,0,False,5257942,en,...,PPL,US,WI,109,76850,None,None,<NA>,United States,1
1738,Johannesburg,Johannesburg,Johannesburg,johannesburg,johannesburg,0,0,False,993800,en,...,PPLA,ZA,06,JHB,JHB,None,None,<NA>,South Africa,1


In [8]:
%%sql
SELECT * FROM allNames WHERE geonameId = 2267057;

Running query in 'duckdb'

geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri
2267057,gsw,Lissabon,None,None,None,None,None
2267057,ady,Лиссабон,None,None,None,None,None
2267057,br,Lisbon,None,None,None,None,None
2267057,ro,Lisabona,None,None,None,None,None
2267057,sc,Lisbona,None,None,None,None,None
2267057,None,Lisboa,False,None,None,None,https://d-nb.info/gnd/4035919-0
2267057,hr,Lisabon,None,None,None,None,None
2267057,no,Lisboa,None,None,None,None,None
2267057,am,ሊዝቦን,None,None,None,None,None
2267057,sr,Лисабон,None,None,None,None,None


In [9]:
%%sql
SELECT levenshtein(nfc_normalize(lower(alternateName)), nfc_normalize(lower('N.Y.'))) AS distance, (CASE WHEN alternateName SIMILAR TO 'Austr\w+' THEN 0 ELSE distance END) AS score, *
FROM allNames NATURAL LEFT JOIN simplifiedGeonames
WHERE (score <= 2) AND feature_class = 'A'
ORDER BY score
LIMIT 10;

Running query in 'duckdb'

distance,score,geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri,name,asciiname,feature_class,feature_code,country_code,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,parentCityId
9,0,2077456,gl,Australia,True,None,None,None,None,Commonwealth of Australia,Commonwealth of Australia,A,PCLI,AU,00,None,None,None,None,None
7,0,2782113,id,Austria,True,None,None,None,None,Republic of Austria,Republic of Austria,A,PCLI,AT,00,None,None,None,None,None
9,0,2077456,io,Australia,None,None,None,None,None,Commonwealth of Australia,Commonwealth of Australia,A,PCLI,AU,00,None,None,None,None,None
9,0,2077456,None,Australia,False,None,None,None,https://d-nb.info/gnd/4003900-6,Commonwealth of Australia,Commonwealth of Australia,A,PCLI,AU,00,None,None,None,None,None
9,0,2077456,None,Australie,False,None,None,None,https://d-nb.info/gnd/4003900-6,Commonwealth of Australia,Commonwealth of Australia,A,PCLI,AU,00,None,None,None,None,None
9,0,2077456,id,Australia,True,None,None,None,None,Commonwealth of Australia,Commonwealth of Australia,A,PCLI,AU,00,None,None,None,None,None
9,0,2077456,en,Australia,True,None,None,None,None,Commonwealth of Australia,Commonwealth of Australia,A,PCLI,AU,00,None,None,None,None,None
10,0,2077456,hr,Australija,True,None,None,None,None,Commonwealth of Australia,Commonwealth of Australia,A,PCLI,AU,00,None,None,None,None,None
13,0,1966436,lv,Austrumtimora,True,None,None,None,None,Democratic Republic of Timor-Leste,Democratic Republic of Timor-Leste,A,PCLI,TL,00,None,None,None,None,None
15,0,2789733,lv,Austrumflandrija,None,None,None,None,None,Provincie Oost-Vlaanderen,Provincie Oost-Vlaanderen,A,ADM2,BE,VLG,VOV,None,None,None,None


In [10]:
%%sql
SELECT COUNT(DISTINCT gnd.geonameId) FROM gndNames NATURAL JOIN gnd
WHERE NOT EXISTS (
    SELECT 1 FROM alternateNames
    WHERE alternateNames.alternateName = gndNames.name AND alternateNames.geonameId = gnd.geonameId
)
LIMIT 10;

Running query in 'duckdb'

count(DISTINCT gnd.geonameId)
45838


There are ADMD entities that are the only reference of admin1_code. For now, ignore.

In [11]:
%%sql
SELECT country.Country, geonames.* FROM geonames 
    JOIN (
        SELECT child.country_code as country_code, ANY_VALUE(child.geonameId) AS chosen_one FROM geonames AS child
        WHERE 
            starts_with(child.feature_code, 'ADMD') AND 
            child.admin1_code IS NOT NULL AND NOT EXISTS(
                SELECT * FROM geonames AS parent 
                WHERE 
                    parent.country_code = child.country_code AND
                    parent.admin1_code = child.admin1_code AND 
                    starts_with(parent.feature_code, 'ADM1')
            )
        GROUP BY child.country_code
    ) ON chosen_one = geonames.geonameId
    JOIN countryInfo AS country ON geonames.country_code = country.ISO


Running query in 'duckdb'

Country,geonameId,name,asciiname,alternatenames,latitude,longitude,feature_class,feature_code,country_code,cc2,admin1_code,admin2_code,admin3_code,admin4_code,admin5_code,population,elevation,dem,timezone,modification_date
Czechia,3061863,Západočeský kraj,Zapadocesky kraj,"Wesbohmisches Gebiet,Wesböhmisches Gebiet,Western Bohemian Region,Zapadocesko,Západočesko",49.66667175292969,13.0,A,ADMD,CZ,None,00,None,None,None,None,0,None,502,Europe/Prague,2008-10-08
French Polynesia,4033093,District de Vairao,District de Vairao,"District de Vairao,Vairao",-17.76667022705078,-149.2833251953125,A,ADMD,PF,None,00,None,None,None,None,0,None,123,Pacific/Tahiti,2012-01-18
Germany,2856312,Ostfalen,Ostfalen,"Eastfalen,Eastphalia,Estfalia,Oostfalen,Ostfaal,Ostfaalen,Ostfalai [a. 775],Ostfalen,Ostfalia,Ostfalija,Ostfalya,Ostfaolen,Oſtfalai [a. 775],Остфалия",52.5,10.833330154418945,A,ADMDH,DE,None,00,None,None,None,None,0,None,76,Europe/Berlin,2016-03-15
Republic of the Congo,2258223,Louessé-Niari,Louesse-Niari,"Bouenza-Louesse,Bouenza-Louessé,Louesse-Niari,Louessé-Niari",-3.0,13.0,A,ADMD,CG,None,00,None,None,None,None,0,None,453,Africa/Brazzaville,2012-01-17
Switzerland,2659040,Ried bei Brig,Ried bei Brig,"Ried bei Brig,Ried-Brig",46.31666946411133,8.016670227050781,A,ADMD,CH,None,00,None,6008,None,None,0,None,903,Europe/Zurich,2012-01-17
Cook Islands,4035452,Vaii,Vaii,Vaii,-21.259469985961914,-159.74383544921875,A,ADMD,CK,None,00,None,None,None,None,0,None,102,Pacific/Rarotonga,2018-06-08
Samoa,4034973,Tuamasaga West,Tuamasaga West,None,-13.833330154418945,-171.86666870117188,A,ADMD,WS,None,00,None,None,None,None,0,None,106,Pacific/Apia,1993-12-22
Poland,753867,Zamojskie Województwo,Zamojskie Wojewodztwo,None,50.75,23.25,A,ADMD,PL,None,00,None,None,None,None,0,None,205,Europe/Warsaw,2007-09-10
Indonesia,1213517,Kabupaten Tapanuli Utara,Kabupaten Tapanuli Utara,"Daerah Tingkat II Tapanuli Utara,Kabupaten Tapanuli Utara",2.25,98.75,A,ADMD,ID,None,00,None,None,None,None,0,None,1405,Asia/Jakarta,2012-01-17
Kyrgyzstan,1527185,Tonskiy Rayon,Tonskiy Rayon,None,42.0,77.0,A,ADMD,KG,None,00,None,None,None,None,0,None,2933,Asia/Bishkek,1994-01-25


There are places under multiple cities in the informal hierachy. For now, include all.

In [12]:
%%sql
SELECT 
    childId AS childId,
    child.name AS childName,
    unnest(list(parentId)) AS parentId,
    unnest(list(parent.name)) AS parentName
FROM hierarchy
JOIN geonames AS parent ON hierarchy.parentId = parent.geonameId
JOIN geonames AS child ON hierarchy.childId = child.geonameId
WHERE 
    parent.feature_class = 'P' AND parent.feature_code != 'PPLX' AND
    hierarchy.type IS DISTINCT FROM 'ADM'
GROUP BY childId, child.name
HAVING COUNT(*) > 1;

Running query in 'duckdb'

childId,childName,parentId,parentName
9972523,Dandenong South,2158177,Melbourne
9972523,Dandenong South,2169455,Dandenong West
7670934,Malatia-Sebastia,616052,Yerevan
7670934,Malatia-Sebastia,11111089,Patvo
377045,Burri Al Drayssah,379252,Khartoum
377045,Burri Al Drayssah,10242960,Burri Al Lamab
847507,Suurmetsä,632453,Vantaa
847507,Suurmetsä,12750226,North Helsinki
641374,Pitäjänmäki,658225,Helsinki
641374,Pitäjänmäki,12750226,North Helsinki


In [13]:
%%sql
SELECT childId, child.name AS childName, child.feature_code, parentId, parent.name AS parentName, parent.admin1_code, parent.admin2_code, parent.feature_code as parentFeatureCode, level FROM fullHierarchy 
    JOIN geonames AS parent ON fullHierarchy.parentId = parent.geonameId 
    JOIN geonames AS child ON fullHierarchy.childId = child.geonameId
WHERE fullHierarchy.childId = 9408119

Running query in 'duckdb'

childId,childName,feature_code,parentId,parentName,admin1_code,admin2_code,parentFeatureCode,level
9408119,Main Ridge,PPLX,2077456,Commonwealth of Australia,00,None,PCLI,Country
9408119,Main Ridge,PPLX,2145234,State of Victoria,07,None,ADM1,ADM1
9408119,Main Ridge,PPLX,2158177,Melbourne,07,24600,PPLA,City
9408119,Main Ridge,PPLX,2166453,Flinders,07,25340,PPL,City
9408119,Main Ridge,PPLX,7839813,Mornington Peninsula,07,25340,ADM2,ADM2


In [14]:
%%sql
SELECT * FROM fullHierarchy JOIN allNames ON fullHierarchy.parentId = allNames.geonameId 
WHERE fullHierarchy.childId = 9408119 
LIMIT 10

Running query in 'duckdb'

childId,parentId,level,geonameId,isolanguage,alternateName,isPreferredName,isShortName,isColloquial,isHistoric,gndUri
9408119,2158177,City,2158177,gre,Μελβούρνη,None,None,None,None,None
9408119,2077456,Country,2077456,sco,Australie,None,None,None,None,None
9408119,2077456,Country,2077456,ru,Австралия,True,None,None,None,None
9408119,2077456,Country,2077456,lt,Australija,True,None,None,None,None
9408119,2145234,ADM1,2145234,fy,Victoria,None,None,None,None,None
9408119,2077456,Country,2077456,vi,Australia (Úc),True,None,None,None,None
9408119,2145234,ADM1,2145234,sd,وڪٽوريا,None,None,None,None,None
9408119,2158177,City,2158177,is,Melbourne,None,None,None,None,None
9408119,2158177,City,2158177,bg,Мелбърн,None,None,None,None,None
9408119,2158177,City,2158177,no,Melbourne,None,None,None,None,None
